# NLP Laboratory Assignment 06 — SMS Spam Detection

**Affan Khan Samid Khan | Roll No. 421135 | Academic Year 2026–27**  
**Faculty:** Dr. Kapil N. Vhatkar

## Aim
Develop and evaluate an NLP system that classifies SMS messages as legitimate (ham) or spam.

## Problem and motivation
Unwanted SMS may contain advertising, deceptive prize claims, premium-rate prompts or malicious links. NLP can learn text patterns such as words, phrases, digits and URLs to support automated screening. The task is binary supervised text classification.

## Dataset
UCI SMS Spam Collection: 5,574 English SMS records; 4,827 ham (86.6%) and 747 spam (13.4%). Source: https://archive.ics.uci.edu/dataset/228/sms+spam+collection. Dataset is historical and class-imbalanced.

## Objectives
1. Inspect and clean labelled SMS data.
2. Normalize text while retaining potentially useful spam indicators.
3. Extract TF-IDF unigram and bigram features.
4. Train Multinomial Naive Bayes and Logistic Regression.
5. Evaluate with accuracy, spam precision, recall and F1.
6. Inspect incorrect predictions and discuss limitations.

## Methodology
Raw SMS → validation/deduplication → lowercase and whitespace normalization → stratified 80:20 split (seed 42) → TF-IDF (1–2 grams, English stop words, min_df=2, max_df=.98, sublinear TF) → model training → held-out evaluation → error analysis.

## Preprocessing
Missing/invalid rows and exact duplicate label-message pairs are removed. Tokenization and n-gram construction are performed by TfidfVectorizer. Stop words are filtered. Stemming/lemmatization is not applied. Digits, punctuation and URLs are retained because they may signal spam.

## Models and configuration
- Multinomial Naive Bayes: alpha=0.5.
- Logistic Regression: C=2.0, class_weight=balanced, max_iter=1000.
- TF-IDF: word unigrams + bigrams, min_df=2, max_df=0.98, sublinear_tf=True.
- Test split: 20%, stratified, random_state=42.
- Classical scikit-learn models; batch size and epochs are not applicable.

## Evaluation
Accuracy measures overall correctness. Spam precision measures reliability of spam alerts; spam recall measures detected spam; spam F1 balances spam precision and recall. Since ham is the majority class, accuracy alone is insufficient. The notebook computes the exact results from the downloaded dataset and saves a comparison table, confusion matrix and error-analysis CSV. **No numerical scores are fabricated in this report; copy the measured values from the executed notebook output.**

## Error analysis
Inspect outputs/error_analysis.csv for actual false positives and false negatives. Possible causes include ambiguous short texts, unseen vocabulary, URL obfuscation, spelling variation and changed spam patterns.

## Advantages
- Lightweight, fast and reproducible.
- Uses open-source Python libraries.
- Compares two standard classifiers.
- Produces class-sensitive metrics and error examples.

## Limitations
- Historical, English-only data.
- Imbalanced classes and evolving scam language.
- TF-IDF has limited contextual understanding.
- Test results may not generalize to current multilingual phishing.

## Ethics and practical considerations
SMS may contain personal information; minimize collection and retention. False positives can hide legitimate messages and false negatives can expose users to scams. Use human review rather than automatic deletion, evaluate language-group fairness, and communicate uncertainty.

## Future scope
Add recent multilingual data, time-based validation, character n-grams, threshold tuning, contextual transformer models and human-in-the-loop feedback.

## Viva answers
1. **Problem:** Binary SMS ham/spam classification.
2. **Why NLP:** Relevant evidence is present in natural-language message text.
3. **Dataset:** UCI SMS Spam Collection, 5,574 English messages.
4. **Preprocessing:** Validation, deduplication, lowercase, whitespace normalization, TF-IDF tokenization and stop-word filtering.
5. **Model:** TF-IDF with Naive Bayes and Logistic Regression; select by spam F1.
6. **Metrics:** Accuracy, spam precision, recall and F1 due to class imbalance.
7. **Errors:** Ambiguity, unseen terms, obfuscated URLs, short context and historical data bias.
8. **Limitations:** Historical English-only corpus and limited context.
9. **Improvements:** Newer data, multilingual support, character n-grams, threshold tuning and transformers.
10. **Ethics:** Privacy, fairness, false alerts and human oversight.

## References
1. Almeida & Hidalgo (2011), SMS Spam Collection, UCI ML Repository, https://doi.org/10.24432/C5CC84
2. https://archive.ics.uci.edu/dataset/228/sms+spam+collection
3. Pedregosa et al. (2011), Scikit-learn: Machine Learning in Python, JMLR 12, 2825–2830.
4. Manning, Raghavan & Schütze (2008), Introduction to Information Retrieval.


In [ ]:
"""NLP Assignment 06: SMS Spam Detection | Affan Khan | Roll 421135
Run: pip install -r requirements.txt && python 421135_Affan_Khan_NLP_Assignment06.py
Downloads the UCI SMS Spam Collection and computes all metrics from a stratified hold-out set.
"""
from pathlib import Path
import io, zipfile, urllib.request
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

ROOT=Path(__file__).resolve().parent
DATA=ROOT/"data"; OUT=ROOT/"outputs"
DATA.mkdir(exist_ok=True); OUT.mkdir(exist_ok=True)
FILE=DATA/"SMSSpamCollection"
URL="https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"

def load_data():
    if not FILE.exists():
        req=urllib.request.Request(URL,headers={"User-Agent":"Mozilla/5.0"})
        with urllib.request.urlopen(req,timeout=60) as r: payload=r.read()
        with zipfile.ZipFile(io.BytesIO(payload)) as z:
            name=next(n for n in z.namelist() if n.endswith("SMSSpamCollection"))
            FILE.write_bytes(z.read(name))
    d=pd.read_csv(FILE,sep="\\t",header=None,names=["label","message"],encoding="utf-8")
    d=d.dropna(subset=["label","message"]).copy()
    d["label"]=d["label"].astype(str).str.strip().str.lower()
    d["message"]=d["message"].astype(str)
    d=d[d.label.isin(["ham","spam"])].drop_duplicates(["label","message"]).reset_index(drop=True)
    return d

def clean(text): return " ".join(str(text).lower().split())

def main():
    d=load_data()
    d["clean"]=d.message.map(clean); d["target"]=d.label.map({"ham":0,"spam":1}).astype(int)
    print("Records:",len(d),"\\nClass counts:\\n",d.label.value_counts())
    print("Missing:",d[["label","message"]].isna().sum().sum(),
          "Duplicates:",d.duplicated(["label","message"]).sum())
    print("Mean/median chars:",round(d.message.str.len().mean(),2),d.message.str.len().median())
    Xtr,Xte,ytr,yte=train_test_split(d.clean,d.target,test_size=.2,random_state=42,stratify=d.target)
    vec=TfidfVectorizer(lowercase=False,stop_words="english",ngram_range=(1,2),min_df=2,max_df=.98,sublinear_tf=True)
    models={
      "Multinomial Naive Bayes":Pipeline([("tfidf",vec),("clf",MultinomialNB(alpha=.5))]),
      "Logistic Regression":Pipeline([("tfidf",vec),("clf",LogisticRegression(C=2,class_weight="balanced",max_iter=1000,random_state=42))])
    }
    rows=[]; fitted={}
    for name,m in models.items():
        m.fit(Xtr,ytr); p=m.predict(Xte); fitted[name]=m
        rows.append({"Model":name,"Accuracy":accuracy_score(yte,p),
          "Precision (spam)":precision_score(yte,p,zero_division=0),
          "Recall (spam)":recall_score(yte,p,zero_division=0),
          "F1-score (spam)":f1_score(yte,p,zero_division=0)})
    results=pd.DataFrame(rows).sort_values(["F1-score (spam)","Recall (spam)"],ascending=False)
    results.to_csv(OUT/"model_comparison.csv",index=False)
    best=results.iloc[0]["Model"]; model=fitted[best]; pred=model.predict(Xte)
    print("\\nRESULTS\\n",results.to_string(index=False))
    print("\\nSELECTED:",best,"\\n",classification_report(yte,pred,target_names=["ham","spam"],zero_division=0))
    cm=confusion_matrix(yte,pred)
    sns.heatmap(cm,annot=True,fmt="d",cmap="Blues",xticklabels=["ham","spam"],yticklabels=["ham","spam"])
    plt.title("Confusion Matrix — "+best); plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.tight_layout()
    plt.savefig(OUT/"confusion_matrix.png",dpi=180); plt.close()
    test=pd.DataFrame({"message":d.loc[Xte.index,"message"].values,
      "actual":yte.map({0:"ham",1:"spam"}).values,
      "predicted":pd.Series(pred,index=yte.index).map({0:"ham",1:"spam"}).values})
    test["correct"]=test.actual==test.predicted
    test.to_csv(OUT/"test_predictions.csv",index=False)
    test.loc[~test.correct].to_csv(OUT/"error_analysis.csv",index=False)
    for msg in ["Congratulations! You have won a free prize. Call now to claim.",
                "Hey, are we still meeting for lunch today?"]:
        print(("SPAM" if model.predict([clean(msg)])[0] else "HAM"),"|",msg)
    print("Outputs saved in",OUT)

main()
